# MedVision AI: Thesis Analysis Notebook
This notebook ports the core logic from the MedVision API for interactive use in Jupyter Lab. It includes preprocessing, CLAHE enhancement, Model Inference, Grad-CAM (XAI), and RAG-based report generation.

### Server Credentials:
- **Host:** 120.125.96.103
- **Port:** 11006
- **SSH:** `ssh -L 11006:localhost:11006 s111340711@120.125.96.103`

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import sys
from IPython.display import Markdown, display

# Ensure project root is in path
sys.path.append(os.path.abspath('..'))

from report_generator import ReportGenerator

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## 1. Constants & Configuration

In [ ]:
MODEL_MAP = {
    "CNN+CLAHE": {"path": "../cnn+clahe/model_cnn_clahe.keras", "layer": "conv_idx_2"},
    "ResNet50": {"path": "../resnet50/model_resnet50.keras", "layer": "conv5_block3_out"},
    "VGG16": {"path": "../vgg16/model_vgg16.keras", "layer": "block5_conv3"},
    "VGG19": {"path": "../vgg19/model_vgg19.keras", "layer": "block5_conv4"}
}

KB_PATH = "../knowledge_base.md"

## 2. Core Diagnostic Functions

In [ ]:
def apply_clahe(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_c = clahe.apply(img)
    return img_c / 255.0

def validate_grayscale(img_bgr):
    """Checks if the image is grayscale or a color photo."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s, _ = cv2.split(hsv)
    mean_saturation = np.mean(s)
    if mean_saturation > 25:
        return False, mean_saturation
    return True, mean_saturation

def make_gradcam_heatmap(img_tensor, model, last_conv_layer_name):
    # Note: Using the complex Grad-CAM logic from main.py designed for nested models
    base_model = None
    base_layer_idx = -1
    for i, layer in enumerate(model.layers):
        if isinstance(layer, tf.keras.Model):
            base_model = layer
            base_layer_idx = i
            break
    
    if base_model:
        internal_grad_model = tf.keras.Model(
            inputs=[base_model.input],
            outputs=[base_model.get_layer(last_conv_layer_name).output, base_model.output]
        )
        with tf.GradientTape() as tape:
            x = model.layers[0](img_tensor)
            conv_outputs, base_output = internal_grad_model(x)
            x = base_output
            for i in range(base_layer_idx + 1, len(model.layers)):
                x = model.layers[i](x)
            predictions = x
            class_index = tf.argmax(predictions[0])
            loss = predictions[:, class_index]
    else:
        grad_model = tf.keras.Model(
            inputs=[model.inputs],
            outputs=[model.get_layer(last_conv_layer_name).output, model.output]
        )
        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(img_tensor)
            class_index = tf.argmax(predictions[0])
            loss = predictions[:, class_index]
            
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img, heatmap, alpha=0.4):
    img_rgb = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    heatmap_res = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255*heatmap_res), cv2.COLORMAP_JET)
    return cv2.addWeighted(img_rgb, 1-alpha, heatmap_color, alpha, 0)

## 3. Execution Pipeline

In [ ]:
def run_full_diagnosis(image_path, model_name):
    # 1. Load Image
    img_color = cv2.imread(image_path)
    if img_color is None:
        print("Error: Could not load image.")
        return
    
    # 2. Validate Grayscale (Mammogram Check)
    is_gray, sat = validate_grayscale(img_color)
    if not is_gray:
        print(f"REJECTED: Image appears to be a color photograph (Saturation: {sat:.2f}). Please use a grayscale mammogram.")
        return
    
    # 3. Preprocess
    img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
    img_resized = cv2.resize(img_gray, (128, 128))
    img_ready = apply_clahe(img_resized)
    img_input = img_ready.reshape(1, 128, 128, 1)
    
    # 4. Model Prediction
    print(f"Loading {model_name}...")
    model = tf.keras.models.load_model(MODEL_MAP[model_name]["path"])
    prediction = model.predict(img_input)
    pred_class = int(np.argmax(prediction[0]))
    confidence = float(prediction[0][pred_class]) * 100
    label = "Cancer" if pred_class == 1 else "Non-Cancer"
    
    # 5. Grad-CAM
    img_tensor = tf.convert_to_tensor(img_input, dtype=tf.float32)
    heatmap = make_gradcam_heatmap(img_tensor, model, MODEL_MAP[model_name]["layer"])
    overlay = overlay_gradcam(img_ready, heatmap)
    
    # 6. Report Generation
    report_gen = ReportGenerator(KB_PATH)
    finding = f"High-intensity activation centers in {model_name}'s target layers indicate focal architectural distortion." if label == "Cancer" else "Diffuse activation with no focal hotspots detected."
    report = report_gen.generate_report(label, confidence, finding)
    
    # 7. Visualization
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.title(f"Preprocessed (CLAHE) - Result: {label}")
    plt.imshow(img_ready, cmap='gray')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title(f"Explainable AI (Grad-CAM) - {confidence:.1f}% Conf.")
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    
    plt.show()
    
    display(Markdown(f"### Final Clinical Report\n---\n{report}"))

## 4. Run Analysis
To use this notebook, upload a mammogram to the server and update `image_path` below.

In [ ]:
# Example usage (uncomment and provide a path when ready)
# run_full_diagnosis('sample_mammogram.jpg', 'CNN+CLAHE')